<a href="https://colab.research.google.com/github/AnnaRudometkina/bigquery-export-ipython-notebooks/blob/master/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B5%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_1_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Установка необходимой библиотеки для подключения к базе данных ClickHouse
!pip install clickhouse-driver  # Установка Python-коннектора для ClickHouse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.2 MB/s eta 0:00:00


In [2]:
# Импортируем необходимые библиотеки
import requests as req  # для выполнения HTTP-запросов
import pandas as pd  # для обработки данных
from datetime import datetime, timedelta  # для работы с датами
import json  # для парсинга json
from clickhouse_driver import Client  # для подключения к ClickHouse

In [3]:
# ============================================
# Параметры API
# ============================================
API_KEY = "043dc9dad696914726d3064e9d917294"
SOURCE_CURRENCY = "USD"
START_DATE = "2023-01-01"
END_DATE = "2023-01-01"

url = f"https://api.exchangerate.host/timeframe?access_key={API_KEY}&source={SOURCE_CURRENCY}&start_date={START_DATE}&end_date={END_DATE}"
s_file = "exchange_rates_raw.json"
csv_file = "exchange_rates_2023_01_01.csv"
TABLE_NAME = "exchange_rates_2023_01_01"

In [4]:
# Подключение к ClickHouse
CH_CLIENT = Client(
    host='158.160.116.58',
    user='student',
    password='dfqh89fhq8',
    database='sandbox'
)

In [5]:
# ============================================
# Шаг 1: Extract
# ============================================
# Функция для извлечения данных с API и сохранения их в локальный файл
def extract_data(url, s_file):
    response = req.get(url)
    if response.status_code == 200:
        data = response.json()
        if data.get('success'):
            with open(s_file, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            print(f"✅ Данные сохранены в {s_file}")
            return data
        else:
            print(f"❌ Ошибка API: {data.get('error', {}).get('info', 'Неизвестная ошибка')}")
            return None
    else:
        print(f"❌ Ошибка HTTP: {response.status_code}")
        return None

In [6]:
# ============================================
# Шаг 2: Transform
# ============================================
# Функция для обработки данных в формате JSON и преобразования их в CSV
def transform_data(s_file, csv_file):
    with open(s_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    source_currency = data.get('source', 'USD')
    quotes = data.get('quotes', {})

    rows = []
    for date, rates in quotes.items():
        for currency_code, rate_value in rates.items():
            if currency_code.startswith(source_currency):
                clean_currency = currency_code[len(source_currency):]
                rows.append({
                    'date': date,
                    'currency_source': source_currency,
                    'currency': clean_currency,
                    'value': rate_value
                })

    df = pd.DataFrame(rows)
    df = df.sort_values(['date', 'currency']).reset_index(drop=True)
    df.to_csv(csv_file, index=False, encoding='utf-8')

    print(f"✅ Данные сохранены в {csv_file}")
    print(f"📊 Всего записей: {len(df)}")
    return df

In [9]:
# ============================================
# Шаг 3: Load (Загрузка в ClickHouse)
# ============================================
# Функция для загрузки данных в ClickHouse из CSV
def upload_to_clickhouse(csv_file, table_name, client):
    """
    Эта функция считывает CSV файл, создает таблицу в
    базе данных ClickHouse и добавляет данные в неё.
    Перед загрузкой таблица очищается (удаляются старые данные).
    """
    # Читаем CSV
    df = pd.read_csv(csv_file)

    if df.empty:
        print("⚠️ CSV файл пуст. Загрузка отменена.")
        return False

    print(f"📊 Загружено {len(df)} записей из CSV")

    # Создаем таблицу (если не существует)
    create_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        date String,
        currency_source String,
        currency String,
        value Float64
    ) ENGINE = Log()
    """

    try:
        client.execute(create_query)
        print(f"✅ Таблица {table_name} создана (или уже существует)")
    except Exception as e:
        print(f"❌ Ошибка при создании таблицы: {e}")
        return False

    # ============================================
    # ПРЕДВАРИТЕЛЬНАЯ ОЧИСТКА ТАБЛИЦЫ
    # ============================================
    try:
        truncate_query = f"TRUNCATE TABLE {table_name}"
        client.execute(truncate_query)
        print(f"🧹 Таблица {table_name} очищена от старых данных")
    except Exception as e:
        print(f"⚠️ Не удалось очистить таблицу: {e}")
        print("   Возможно, таблица пуста или используется другой движок.")
        print("   Продолжаем загрузку...")

    # Подготавливаем данные для вставки
    data_to_insert = list(df.itertuples(index=False, name=None))
    insert_query = f"INSERT INTO {table_name} (date, currency_source, currency, value) VALUES"

    try:
        client.execute(insert_query, data_to_insert)
        print(f"✅ Данные успешно загружены в таблицу {table_name}")
        print(f"📊 Загружено записей: {len(data_to_insert)}")
        return True
    except Exception as e:
        print(f"❌ Ошибка при вставке данных: {e}")
        return False

In [10]:
# ============================================
# Запуск ETL-процесса
# ============================================
print("🚀 Начинаем ETL-процесс...")
print("=" * 50)

print("\n📥 Шаг 1: Извлечение данных из API...")
raw_data = extract_data(url, s_file)

if raw_data:
    print("\n🔄 Шаг 2: Трансформация данных...")
    df = transform_data(s_file, csv_file)

    print("\n📤 Шаг 3: Загрузка данных в ClickHouse...")
    upload_to_clickhouse(csv_file, TABLE_NAME, CH_CLIENT)

    print("\n📋 Проверка данных в ClickHouse:")
    result = CH_CLIENT.execute(f"SELECT * FROM {TABLE_NAME} LIMIT 5")
    for row in result:
        print(row)

    count_result = CH_CLIENT.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
    print(f"\n📊 Всего записей в таблице: {count_result[0][0]}")
else:
    print("❌ Ошибка при извлечении данных. Процесс остановлен.")

🚀 Начинаем ETL-процесс...

📥 Шаг 1: Извлечение данных из API...
✅ Данные сохранены в exchange_rates_raw.json

🔄 Шаг 2: Трансформация данных...
✅ Данные сохранены в exchange_rates_2023-01-01.csv
📊 Всего записей: 169

📤 Шаг 3: Загрузка данных в ClickHouse...
📊 Загружено 169 записей из CSV
✅ Таблица exchange_rates_2023_01_01 создана (или уже существует)
🧹 Таблица exchange_rates_2023_01_01 очищена от старых данных
✅ Данные успешно загружены в таблицу exchange_rates_2023_01_01
📊 Загружено записей: 169

📋 Проверка данных в ClickHouse:
('2023-01-01', 'USD', 'AED', 3.672635)
('2023-01-01', 'USD', 'AFN', 87.49408)
('2023-01-01', 'USD', 'ALL', 107.150283)
('2023-01-01', 'USD', 'AMD', 393.731324)
('2023-01-01', 'USD', 'ANG', 1.802385)

📊 Всего записей в таблице: 169
